# Pooled MA-IDM

In [ ]:
import arviz as az
import numpy as np
"""
# If pymc3
"""
# import pymc3 as pm
# from theano import tensor as tt
"""
# If pymc4
"""
import pymc as pm
import aesara.tensor as tt

import random
import os
import sys
import matplotlib.pyplot as plt
plt.rcParams['text.usetex'] = True

import pickle

from os import path

import warnings
warnings.filterwarnings("ignore")

np.random.seed(1116)

class Config:
    frame_rate = 5
    dt = 0.2
    

In [ ]:
def formalize_array(x, step, slice_len, skip, sliding_window=True):
    if sliding_window:
        N = int(np.floor((x.shape[0]-slice_len*step+1)/skip))
        x_all = np.zeros((N,slice_len))
        for i in range(N):
            x_all[i,:] = x[i*skip:i*skip+slice_len*step:step]
    else:
        x = x[:: step]
        N = int(np.floor(x.shape[0]/slice_len))
        x_all = np.zeros((N,slice_len))
        for i in range(N):
            x_all[i,:] = x[slice_len*i:slice_len*(i+1)]
    return x_all

def load_training_data(step, skip, pair_id_list):
    # Load the data
    with open('./filtered_car_following_pairs.pkl', 'rb') as f:
        tracks = pickle.load(f)
    slice_len=int(4/(step*Config.dt))
    for pair_id in pair_id_list:
        if pair_id == pair_id_list[0]:
            xt = formalize_array(tracks[pair_id]['follower_x'][:-1], step, slice_len, skip)
            vt = formalize_array(tracks[pair_id]['follower_speed'][:-1], step, slice_len, skip)
            s = formalize_array(tracks[pair_id]['gap'][:-1], step, slice_len, skip)
            dv = formalize_array(tracks[pair_id]['relative_speed'][:-1], step, slice_len, skip)
            label_v = formalize_array(tracks[pair_id]['follower_speed'][1:], step, slice_len, skip)
            label_x = formalize_array(tracks[pair_id]['follower_x'][1:], step, slice_len, skip)
            
        else:
            xt = np.vstack((xt,formalize_array(tracks[pair_id]['follower_x'][:-1], step, slice_len, skip)))
            vt = np.vstack((vt,formalize_array(tracks[pair_id]['follower_speed'][:-1], step, slice_len, skip)))
            s = np.vstack((s,formalize_array(tracks[pair_id]['gap'][:-1], step, slice_len, skip)))
            dv = np.vstack((dv, formalize_array(tracks[pair_id]['relative_speed'][:-1], step, slice_len, skip)))
            label_v = np.vstack((label_v, formalize_array(tracks[pair_id]['follower_speed'][1:], step, slice_len, skip)))
            label_x = np.vstack((label_x, formalize_array(tracks[pair_id]['follower_x'][1:], step, slice_len, skip)))
            
    print("ID list:", pair_id_list, ", Data size:", label_v.shape)
    return xt, vt, s, dv, label_v, label_x

def MA_IDM_pool(base_path, step):
    # load interactive data for car
    xt, vt, s, dv, label_v, label_x = load_training_data(step, skip=5, pair_id_list=range(500))
    
    labels = np.hstack((label_v, label_x))
    print("training size:", labels.shape)
    
    dt = Config.dt

    model = pm.Model()

    D = 5
    
    slice_len=int(4/(step*Config.dt))
    GP_t = np.array(range(slice_len*step))[::step].reshape(-1,1)
    shh = GP_t.shape[0]
    
    with model:
        def MA_IDM_xv(VMAX, DSAFE, TSAFE, AMAX, AMIN, DELTA, s, vt, dv, xt):
            sn = DSAFE + vt * TSAFE + vt * dv / (2 * tt.sqrt(AMAX * AMIN))
            a = AMAX * (1 - (vt / VMAX) ** DELTA - (sn / s) ** 2)
            vt_next = vt + a * Config.dt
            return tt.concatenate([vt_next, xt + 0.5 * (vt_next + vt) * Config.dt], axis=1)
        
        s_ = pm.ConstantData("s", s)
        xt_ = pm.ConstantData("xt", xt)
        vt_ = pm.ConstantData("vt", vt)
        dv_ = pm.ConstantData("dv", dv)
        
        mu_prior = pm.floatX(np.array([0,0,0,0,0]))
        parameters_normalized = pm.MvNormal('mu_normalized', mu_prior, chol=np.eye(D))
        
        log_parameters = pm.Deterministic('log_mu', parameters_normalized*np.array([.1, .1, .1, .1, .1])
                                      +np.array([2., 0.69, 0.47, .4, .51]))
        
        parameters = pm.Deterministic('mu', tt.exp(log_parameters))
        
        DELTA = 4 
        
        #################################################
        # priors on the covariance function hyperparameters
        l = pm.Normal('l', mu=Config.frame_rate, sigma=Config.frame_rate)
        # l = pm.Normal('l', mu=1*Config.frame_rate, sigma=1*Config.frame_rate)
        
        # prior on the function variance
        s_f = pm.Exponential('s_f', lam=1e4) # 1e3
        s2_f = pm.Deterministic('s2_f', s_f**2)
        #################################################
        
        # covariance functions for the function f and the noise
        cov_func = pm.gp.cov.ExpQuad(1, l)
        cov_t = tt.stacklists([[dt ** 2, 0.5 * dt ** 3],
                               [0.5 * dt ** 3, 1/4 * dt ** 4]])
        s2_v = pm.Exponential('s2_v', lam=1e5)
        s2_x = pm.Exponential('s2_x', lam=1e3)
        
        cov_noise = tt.slinalg.kron(tt.stacklists([[s2_v, 0], [0, s2_x]]), tt.eye(shh))
        
        mu = MA_IDM_xv(parameters[0], parameters[1], parameters[2],
                       parameters[3], parameters[4],DELTA, s_, vt_, dv_, xt_)
        
        v_obs = pm.MvNormal('obs', mu=mu, cov=tt.slinalg.kron(cov_t, s2_f*cov_func(GP_t)) + cov_noise, observed=labels)
        
        tr = pm.sample(draws=2500, tune=3000, random_seed=16, init='jitter+adapt_diag_grad', chains=1,
                       cores=8, discard_tuned_samples=True, return_inferencedata=True, target_accept=0.90)
    return tr, model

In [ ]:
base_path = '../data/highD/'
step = 1
tr, model = MA_IDM_pool(base_path, step)

In [ ]:
import pickle
from pickle import UnpicklingError

cache = "./MA_IDM_pooled_car_I24-motion.pkl"
if path.exists(cache):
    try:
        fp = open(cache, 'rb')
        tr = pickle.load(fp)
        fp.close()
        print("Load trace", cache, ": done!")
    except UnpicklingError:
        os.remove(cache)
        print('Removed broken cache:', cache)
else:
    output_file = open(cache, 'wb')
    pickle.dump(tr, output_file)
    output_file.close()
    print("Generated and Saved", output_file, ": done!")

In [ ]:
print(np.sqrt(tr.posterior.s2_f.mean(axis=0).mean(axis=0)))
print(np.sqrt(tr.posterior.s2_x.mean(axis=0).mean(axis=0)))
print(np.sqrt(tr.posterior.s2_v.mean(axis=0).mean(axis=0)))
print(tr.posterior.l.mean(axis=0).mean(axis=0)* Config.dt)
print(tr.posterior.l.mean(axis=0).mean(axis=0))

In [ ]:
_ = az.plot_trace(tr, var_names=["mu"], coords={"mu_dim_0":0},compact=True)
_ = az.plot_trace(tr, var_names=["mu"], coords={"mu_dim_0":1}, compact=True)
_ = az.plot_trace(tr, var_names=["mu"], coords={"mu_dim_0":2}, compact=True)
_ = az.plot_trace(tr, var_names=["mu"], coords={"mu_dim_0":3}, compact=True)
_ = az.plot_trace(tr, var_names=["mu"], coords={"mu_dim_0":4}, compact=True)
_ = az.plot_trace(tr, var_names=["l","s2_f"],figsize=(15,8), compact=True)

In [ ]:
az.summary(tr, var_names=["mu","log_mu", "l", "s2_f"])

In [ ]:
import corner
import matplotlib

label_list = [r'$v_0$',r'$s_0$', r'$T$', r'$\alpha$', r'$\beta$']
fontsize = 18
matplotlib.rc('xtick', labelsize=fontsize) 
matplotlib.rc('ytick', labelsize=fontsize) 

figure = corner.corner(
    tr,
    var_names=['mu'],
    smooth=1.8,
    color = 'k',
    plot_contours=True,
    plot_density=False,
    plot_datapoints = False,
    bins = 20,
    show_titles=True,
    labels=label_list,
    reverse=False,
)

ax_new = figure.add_axes([.66, .66, .27, .27])
cov = np.cov(tr.posterior.mu[0,:,:], rowvar=False)
Dinv = np.diag(1 / np.sqrt(np.diag(cov)))
corr = Dinv @ cov @ Dinv
kwargs = {'cmap':'plasma','interpolation':'nearest', 'vmin':-1}
corr_show = ax_new.matshow(corr, **kwargs)
c_bar = figure.colorbar(corr_show, ax=ax_new, extend='both')
ax_new.set_xticklabels(['']+label_list)
ax_new.set_yticklabels(['']+label_list)
ax_new.set_title('Correlation Matrix')
for item in ([ax_new.title, ax_new.xaxis.label, ax_new.yaxis.label] +
             ax_new.get_xticklabels() + ax_new.get_yticklabels()):
    item.set_fontsize(fontsize)

# figure.savefig('../Figs/GP_IDM_pool_car.pdf', dpi=300)

In [ ]:
# import corner

_ = corner.corner(
    tr,
    smooth=1.8,
    plot_density=False,
    bins=30,
    var_names=["l", "s2_f"]
)